# 01 · Build the commonsense-alignment sample

Builds the fixed experimental input for H1/H1b/H2:

- **Commonsense / anticommonsense pairs**: the same formal question (story, graph, query type, given probabilities, answer) with one variable renamed. Matched one to one.
- **Noncommonsense items**: invented variable names, different parameters, compared as a group.

Each variant contributes `SIZE_PER_VARIANT` items, stratified by rung and answer, with the documented seed. The prompt text includes the meta-model `background`, which is where CLadder states the causal graph.

The seed is created with a random number. 

Notebook 02 rebuilds the same sample on Colab and checks the SHA-256 against the manifest written here.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import hashlib
import json

import pandas as pd

from causal_evaluation.data.variants import (
    build_pairs,
    build_variant_sample,
    load_variant,
    write_jsonl,
)
from causal_evaluation.models.logprob import item_text

RAW = ROOT / "data" / "raw"
SEED = 20260903
SIZE_PER_VARIANT = 1000
SAMPLE_PATH = Path(os.environ.get("CE_INTERIM", ROOT / "data" / "interim")) / "variants-sample.jsonl"

## Matched pairs available

In [ ]:
pairs = build_pairs(load_variant(RAW, "commonsense"), load_variant(RAW, "anticommonsense"))
print(f"{len(pairs):,} one-to-one commonsense/anticommonsense pairs")

## Sample

In [ ]:
items = build_variant_sample(RAW, SIZE_PER_VARIANT, SEED)
sample_sha = write_jsonl(items, SAMPLE_PATH)
sample = pd.DataFrame(items)
print(f"{len(items):,} items -> {SAMPLE_PATH}\nSHA-256 {sample_sha}")
pd.crosstab([sample["variant"], sample["rung"]], sample["answer"], margins=True)

In [ ]:
pd.crosstab(sample["query_type"], sample["variant"])

## Manifest (for reproducibility and the Colab check)

In [ ]:
def file_sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


manifest = {
    "seed": SEED,
    "size_per_variant": SIZE_PER_VARIANT,
    "n_items": len(items),
    "sample_sha256": sample_sha,
    "raw_sha256": {
        p.name: file_sha(p)
        for p in sorted(RAW.glob("cladder-v1-*.json"))
        if "commonsense" in p.name or "meta-models" in p.name
    },
}
SAMPLE_PATH.with_suffix(".manifest.json").write_text(json.dumps(manifest, indent=2))
manifest

## One matched pair and one nonsense item, as the model sees them

In [ ]:
pair_id = sample.loc[sample["query_type"] == "ate", "pair_id"].iloc[0]
for item in sample[sample["pair_id"] == pair_id].to_dict("records"):
    print(f"--- {item['variant']} (answer: {item['answer']})\n{item_text(item)}\n")
nonsense = sample[sample["variant"] == "noncommonsense"].iloc[0].to_dict()
print(f"--- noncommonsense (answer: {nonsense['answer']})\n{item_text(nonsense)}")